# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv('ames_housing.csv')
df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.
import pandas as pd
from scipy import stats

df = pd.read_csv("ames_housing.csv")

# 1. 품질 점수 5, 6, 7 집단에서 각각 20개 표본 추출
g5 = df.loc[df["OverallQual"]==5, "SalePrice"].dropna().sample(n=20, random_state=5)
g6 = df.loc[df["OverallQual"]==6, "SalePrice"].dropna().sample(n=20, random_state=5)
g7 = df.loc[df["OverallQual"]==7, "SalePrice"].dropna().sample(n=20, random_state=5)

groups = {"5": g5, "6": g6, "7": g7}

# 2 & 4. 표본 수, 평균, 표준편차, 정규성 검정 출력
shapiro_results = {}
for name, g in groups.items():
    shapiro_results[name] = stats.shapiro(g)
    print(f"품질 {name}: n={len(g)}, 평균={g.mean():.2f}, 표준편차={g.std(ddof=1):.2f}, "
          f"정규성 p-value={shapiro_results[name].pvalue:.4f}")

# 3. 독립성 설명 (주석)
# 각 표본은 OverallQual 값이 5, 6, 7인 서로 다른 주택에서 추출되었고,
# 한 주택은 하나의 OverallQual 값만 가지므로 세 집단 사이에 겹치는 주택이 없습니다.
# 즉 반복측정이나 짝지어진 구조가 아니라 서로 다른 대상들을 비교하는 것이므로
# 세 집단은 독립집단(independent groups)입니다.
print("\n독립성: 세 집단은 서로 다른 주택으로 구성된 독립집단입니다.")

# 5. 등분산성 검정 (Levene)
levene_result = stats.levene(g5, g6, g7)
print(f"\n등분산성 p-value: {levene_result.pvalue:.4f}")

# 6. 가설 (주석)
# H0: 세 집단의 모집단 평균 판매가격은 모두 같다.
# H1: 적어도 한 집단의 모집단 평균 판매가격은 다르다.

# 7. 일원배치 ANOVA 수행
f_result = stats.f_oneway(g5, g6, g7)

# 8. F통계량, p-value 출력 및 판단
alpha = 0.05
print(f"F통계량: {f_result.statistic:.4f}")
print(f"p-value: {f_result.pvalue:.4e}")   # .8f 대신 .4e 사용

if f_result.pvalue < alpha:
    print(f"\n전체 판단: p-value({f_result.pvalue:.8f}) < 유의수준({alpha})")
    print("→ 귀무가설 기각: 적어도 한 집단의 평균 판매가격은 다른 집단과 차이가 있습니다.")
else:
    print(f"\n전체 판단: p-value({f_result.pvalue:.8f}) >= 유의수준({alpha})")
    print("→ 귀무가설 기각 실패: 세 집단의 평균 판매가격이 다르다고 볼 근거가 부족합니다.")

# 9. 사후검정 필요 여부
if f_result.pvalue < alpha:
    print("\n사후검정: 필요함 (ANOVA는 '적어도 하나는 다르다'만 알려줄 뿐, "
          "구체적으로 어느 집단과 어느 집단이 다른지는 알려주지 않기 때문에 "
          "Tukey HSD 등 사후검정이 추가로 필요합니다.)")
else:
    print("\n사후검정: 불필요 (전체적으로 유의한 차이가 없다고 판단되었기 때문입니다.)")

품질 5: n=20, 평균=130605.00, 표준편차=24937.11, 정규성 p-value=0.7760
품질 6: n=20, 평균=167826.60, 표준편차=41944.55, 정규성 p-value=0.4096
품질 7: n=20, 평균=217593.60, 표준편차=48298.39, 정규성 p-value=0.1290

독립성: 세 집단은 서로 다른 주택으로 구성된 독립집단입니다.

등분산성 p-value: 0.0792
F통계량: 24.2456
p-value: 2.4031e-08

전체 판단: p-value(0.00000002) < 유의수준(0.05)
→ 귀무가설 기각: 적어도 한 집단의 평균 판매가격은 다른 집단과 차이가 있습니다.

사후검정: 필요함 (ANOVA는 '적어도 하나는 다르다'만 알려줄 뿐, 구체적으로 어느 집단과 어느 집단이 다른지는 알려주지 않기 때문에 Tukey HSD 등 사후검정이 추가로 필요합니다.)


### 필수 1 답변 작성란

- **Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
<br> -> 검정을 반복할수록 전체 분석에서 한 번 이상 제 1종 오류가 발생할 확률이 0.05보다 커지는 다중 비교 문제가 발생한다.

- **Q2.** F통계량은 어떤 두 변동의 비율인가요?  
<br> -> 집단 간 평균 차이를 나타내는 집단 간 변동률을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

- **Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
<br> -> 있습니다. 따라서 적어도 한 집단의 모집단 평균 판매가격은 다르다.

- **Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?
<br> -> 알 수 없음, 구체적인 집단 쌍은 Tukey HSD와 같은 사후 검정으로 확인해야한다.

---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [5]:
# 필수 2 코드를 작성하세요.
g5 = df.loc[df["OverallQual"]==5, "SalePrice"].dropna().sample(n=20, random_state=5)
g6 = df.loc[df["OverallQual"]==6, "SalePrice"].dropna().sample(n=20, random_state=5)
g7 = df.loc[df["OverallQual"]==7, "SalePrice"].dropna().sample(n=20, random_state=5)

# 1. 세 집단 결합
anova_df = pd.concat([
    pd.DataFrame({"SalePrice": g5, "OverallQual": "5"}),
    pd.DataFrame({"SalePrice": g6, "OverallQual": "6"}),
    pd.DataFrame({"SalePrice": g7, "OverallQual": "7"}),
], ignore_index=True)

# 2. 전체 평균
grand_mean = anova_df["SalePrice"].mean()
print("전체 평균(grand mean): ", grand_mean)

groups = {"5": g5, "6": g6, "7": g7}
k = len(groups)
N = len(anova_df)

# 3. ANOVA 표 계산
SS_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups.values())
SS_within = sum(((g - g.mean())**2).sum() for g in groups.values())

df_between = k - 1
df_within = N - k

MS_between = SS_between / df_between
MS_within = SS_within / df_within

F_manual = MS_between / MS_within

print("\n===== ANOVA 표 =====")
print(f"{'요인':<10}{'제곱합(SS)':>18}{'자유도(df)':>12}{'평균제곱(MS)':>18}")
print(f"{'집단 간':<10}{SS_between:>18.4e}{df_between:>12}{MS_between:>18.4e}")
print(f"{'집단 내':<10}{SS_within:>18.4e}{df_within:>12}{MS_within:>18.4e}")
print(f"{'F통계량':<10}{F_manual:>18.4f}")

# 4. stats.f_oneway와 비교
f_result = stats.f_oneway(g5, g6, g7)

print("\n===== F통계량 대조 =====")
print("직접 계산한 F: ", F_manual)
print("f_oneway F: ", f_result.statistic)
print("일치 여부: ", round(F_manual, 6) == round(f_result.statistic, 6))
print("ANOVA p-value: ", f_result.pvalue)

# 5. p-value 조건에 따라 Tukey HSD 실행
alpha = 0.05

if f_result.pvalue < alpha:
    print(f"\np-value({f_result.pvalue:.4e}) < 유의수준({alpha}) -> 귀무가설 기각")
    print("적어도 한 집단 쌍의 평균 판매가격이 다르므로 Tukey HSD 사후검정을 진행합니다.")

    tukey_result = pairwise_tukeyhsd(
        endog=anova_df["SalePrice"],
        groups=anova_df["OverallQual"],
        alpha=alpha
    )

    print("\n===== Tukey HSD 결과 =====")
    print(tukey_result)

    # 6. reject=True인 쌍 확인 + 7. 평균 차이 계산
    means = {name: g.mean() for name, g in groups.items()}

    print("\n===== 집단 쌍별 유의성 및 평균 차이 =====")
    pair_diffs = []
    for row in tukey_result.summary().data[1:]:
        group1, group2, meandiff, p_adj, lower, upper, reject = row
        diff = means[group2] - means[group1]
        pair_diffs.append((group1, group2, diff, reject))

        if reject:
            print(f"품질 {group1} vs 품질 {group2}: 평균 차이 {diff:.2f} -> 유의한 차이 있음 (reject=True)")
        else:
            print(f"품질 {group1} vs 품질 {group2}: 평균 차이 {diff:.2f} -> 유의한 차이 없음 (reject=False)")

    # 8. 가장 차이가 큰 집단 쌍
    max_pair = max(pair_diffs, key=lambda x: abs(x[2]))
    print(f"\n가장 차이가 큰 집단 쌍: 품질 {max_pair[0]} vs 품질 {max_pair[1]} (평균 차이 약 {max_pair[2]:.2f}달러)")

else:
    print(f"\np-value({f_result.pvalue:.4e}) >= 유의수준({alpha}) -> 귀무가설 기각 실패")
    print("전체적으로 유의한 차이가 없으므로 Tukey HSD 사후검정은 수행하지 않습니다.")

전체 평균(grand mean):  172008.4

===== ANOVA 표 =====
요인                   제곱합(SS)     자유도(df)          평균제곱(MS)
집단 간              7.6195e+10           2        3.8097e+10
집단 내              8.9565e+10          57        1.5713e+09
F통계량                 24.2456

===== F통계량 대조 =====
직접 계산한 F:  24.245575326476107
f_oneway F:  24.24557532647611
일치 여부:  True
ANOVA p-value:  2.4031260275435734e-08

p-value(2.4031e-08) < 유의수준(0.05) -> 귀무가설 기각
적어도 한 집단 쌍의 평균 판매가격이 다르므로 Tukey HSD 사후검정을 진행합니다.

===== Tukey HSD 결과 =====
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------

===== 집단 쌍별 유의성 및 평균 차이 =====
품질 5 vs 품질 6: 평균 차이 37221.60 -> 유의한 차이 

### 필수 2 답변 작성란

- **Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
<br> -> SS_between은 집단 평균들이 전체 평균에서 벗어난 집단 간 변동 값
<br> -> SS_within은 각 관측값이 소속 집단 평균에서 벗어난 집단 내 변동

- **Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
<br> -> 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미

- **Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
<br> -> 5-6, 5-7, 6-7의 모든 집단 쌍에서 유의한 차이가 확인된다.

- **Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
<br> -> 품질 5점과 7점 집단이며 평균 차이는 약 86,988달러이다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [ ]:
# 과제 코드를 작성하세요.

### 과제 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

---

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
<br> -> 각 검정마다 위양성 가능성이 있기 때문에 비교 횟수가 늘어날수록 전체  분석에서 한 번 이상 잘못 기각할 확률이 누적이다. 
2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
<br> -> 귀무가설은 모든 집단의 모집단과 평균이 같다. 대립가설은 적어도 한 집단의 평균이 다르다는 것
3. F통계량이 크다는 것은 무엇을 의미하나요?
<br> -> 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미
4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
<br> -> ANOVA는 적어도 한 집단이 다르다는 사실만 알려주며 구체적으로 집단 쌍은 알려주지 않음
5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
<br> -> 비교한 집단 쌍, 평균 차이, 조정된 P-value, 신뢰구간, reject 여부 